# Injury Detection Model — Robot Dog Search & Rescue

Classifies detected people as **OK** (standing/upright) or **POTENTIALLY INJURED** (lying/fallen).

**Pipeline:**
1. Download real labeled images from Kaggle (fall vs not-fall)
2. Run YOLOv8-pose on them to extract real noisy keypoints
3. Engineer geometric features from keypoints
4. Train a Gradient Boosting classifier on those features
5. Supplement with synthetic keypoint augmentation for robustness

In [ ]:
!pip install -q ultralytics scikit-learn opencv-python-headless joblib kagglehub

In [ ]:
import numpy as np
import os
import json
import joblib
from pathlib import Path
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

np.random.seed(42)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

## 1. Download Datasets from Kaggle

We download **two** datasets for maximum coverage:
1. `patsham/fall-image-ai` — classification-style with `Fall/` and `Not Fall/` folders (~6k images)
2. `uttejkumarkandagatla/fall-detection-dataset` — YOLO-format as backup

The more data we feed to YOLOv8-pose, the better our keypoint features will represent real-world variation.

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'charliearam'
os.environ['KAGGLE_KEY'] = 'KGAT_876d88bead9ec9798df6113bad8f8de8'

import kagglehub

# Dataset 1: Classification-style (Fall / Not Fall folders)
print('Downloading patsham/fall-image-ai...')
ds1_path = kagglehub.dataset_download('patsham/fall-image-ai')
print(f'  -> {ds1_path}')

# Dataset 2: YOLO-format fall detection
print('Downloading uttejkumarkandagatla/fall-detection-dataset...')
ds2_path = kagglehub.dataset_download('uttejkumarkandagatla/fall-detection-dataset')
print(f'  -> {ds2_path}')

print('\nDone.')

In [ ]:
# Show full directory tree for both datasets so we can see the structure
def show_tree(root, max_depth=4):
    print(f'\n=== {root} ===')
    for dirpath, dirnames, filenames in os.walk(root):
        level = dirpath.replace(root, '').count(os.sep)
        if level >= max_depth:
            continue
        indent = '  ' * level
        img_count = sum(1 for f in filenames if Path(f).suffix.lower() in IMAGE_EXTS)
        txt_count = sum(1 for f in filenames if f.endswith('.txt'))
        other_count = len(filenames) - img_count - txt_count
        parts = []
        if img_count: parts.append(f'{img_count} images')
        if txt_count: parts.append(f'{txt_count} txt')
        if other_count: parts.append(f'{other_count} other')
        summary = ', '.join(parts) if parts else 'empty'
        print(f'{indent}{os.path.basename(dirpath)}/  [{summary}]')
        # Show a few sample filenames at leaf dirs
        if level == max_depth - 1 or not dirnames:
            sample = filenames[:3]
            for s in sample:
                print(f'{indent}  {s}')
            if len(filenames) > 3:
                print(f'{indent}  ... ({len(filenames)} total)')

show_tree(ds1_path)
show_tree(ds2_path)

## 2. Collect and Label Images

We scan both datasets and map everything to our binary labels:
- **OK (label 0):** standing, sitting, walking, bending, not-fall, notfall
- **INJURED (label 1):** fall, fallen, lying, lying_down, crawling

Strategy:
1. **Folder-per-class** — if a folder name matches a keyword, all images in it get that label
2. **YOLO annotations** — if `.txt` label files exist alongside images, parse the class IDs
3. **Filename patterns** — if filenames contain class keywords

In [ ]:
INJURED_WORDS = {'fall', 'fallen', 'lying', 'lying_down', 'crawling', 'crawl'}
OK_WORDS = {'not_fall', 'notfall', 'not fall', 'no_fall', 'nofall',
            'standing', 'stand', 'sitting', 'sit', 'bending', 'bend',
            'walking', 'walk', 'normal', 'adl', 'daily'}

def classify_name(name):
    """Classify a folder/file name as 'ok', 'injured', or None."""
    n = name.lower().strip().replace('-', '_').replace(' ', '_')
    # Check OK first (since 'not_fall' contains 'fall')
    for w in OK_WORDS:
        if w.replace(' ', '_') in n:
            return 'ok'
    for w in INJURED_WORDS:
        if w in n:
            return 'injured'
    return None


def collect_images_from_dataset(root):
    """Scan a dataset root and return (ok_paths, injured_paths) using multiple strategies."""
    ok, injured = [], []

    for dirpath, dirnames, filenames in os.walk(root):
        folder_name = os.path.basename(dirpath)
        folder_label = classify_name(folder_name)
        imgs_in_dir = [os.path.join(dirpath, f) for f in filenames
                       if Path(f).suffix.lower() in IMAGE_EXTS]

        if not imgs_in_dir:
            continue

        # Strategy 1: folder name tells us the class
        if folder_label == 'ok':
            ok.extend(imgs_in_dir)
            continue
        elif folder_label == 'injured':
            injured.extend(imgs_in_dir)
            continue

        # Strategy 2: YOLO annotations (.txt files alongside images)
        txt_files = {Path(f).stem: os.path.join(dirpath, f)
                     for f in filenames if f.endswith('.txt') and f != 'classes.txt'}
        classes_file = os.path.join(dirpath, 'classes.txt')
        yaml_files = [os.path.join(dirpath, f) for f in filenames
                      if f.endswith('.yaml') or f.endswith('.yml')]

        # Try to find class name mapping
        class_names = {}
        if os.path.exists(classes_file):
            with open(classes_file) as f:
                for i, line in enumerate(f):
                    class_names[i] = line.strip()
        for yf in yaml_files:
            try:
                import yaml
                with open(yf) as f:
                    data = yaml.safe_load(f)
                if 'names' in data:
                    if isinstance(data['names'], list):
                        class_names = {i: n for i, n in enumerate(data['names'])}
                    elif isinstance(data['names'], dict):
                        class_names = data['names']
            except:
                pass

        if txt_files and class_names:
            for img_path in imgs_in_dir:
                stem = Path(img_path).stem
                if stem in txt_files:
                    try:
                        with open(txt_files[stem]) as f:
                            lines = f.readlines()
                        if lines:
                            cls_id = int(lines[0].split()[0])
                            cls_name = class_names.get(cls_id, '')
                            lbl = classify_name(cls_name)
                            if lbl == 'ok':
                                ok.append(img_path)
                            elif lbl == 'injured':
                                injured.append(img_path)
                    except:
                        pass
            continue

        # Strategy 3: filename contains class keyword
        for img_path in imgs_in_dir:
            fname_label = classify_name(Path(img_path).stem)
            if fname_label == 'ok':
                ok.append(img_path)
            elif fname_label == 'injured':
                injured.append(img_path)

    return ok, injured


ok1, inj1 = collect_images_from_dataset(ds1_path)
ok2, inj2 = collect_images_from_dataset(ds2_path)

ok_paths = ok1 + ok2
injured_paths = inj1 + inj2

print(f'Dataset 1 (fall-image-ai):       OK={len(ok1):>5}, INJURED={len(inj1):>5}')
print(f'Dataset 2 (fall-detection):       OK={len(ok2):>5}, INJURED={len(inj2):>5}')
print(f'-----------------------------------------------')
print(f'TOTAL:                            OK={len(ok_paths):>5}, INJURED={len(injured_paths):>5}')

if len(ok_paths) == 0 or len(injured_paths) == 0:
    print('\n*** WARNING: One class is empty! ***')
    print('Check the directory tree output above. You may need to edit')
    print('INJURED_WORDS or OK_WORDS to match the folder/file names.')
    print('\nSample folder names found:')
    for root_path in [ds1_path, ds2_path]:
        for d in sorted(Path(root_path).rglob('*')):
            if d.is_dir():
                imgs = sum(1 for f in d.iterdir() if f.suffix.lower() in IMAGE_EXTS)
                if imgs > 0:
                    print(f"  '{d.name}' -> {imgs} images (classify_name='{classify_name(d.name)}')")    

assert len(ok_paths) > 0 and len(injured_paths) > 0, \
    f'Need images in both classes. OK={len(ok_paths)}, INJURED={len(injured_paths)}'

In [ ]:
# Preview samples
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Sample Images', fontsize=14)

for i, path in enumerate(ok_paths[:5]):
    img = cv2.imread(path)
    if img is not None:
        axes[0, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f'OK #{i+1}')
    axes[0, i].axis('off')

for i, path in enumerate(injured_paths[:5]):
    img = cv2.imread(path)
    if img is not None:
        axes[1, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[1, i].set_title(f'INJURED #{i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 3. Extract Real Keypoints with YOLOv8-Pose

Run YOLOv8-pose on every image to extract real, noisy keypoints — the same kind the model will see at inference time. This takes a few minutes on T4.

In [ ]:
pose_model = YOLO('yolov8s-pose.pt')
print('YOLOv8-pose loaded.')

In [ ]:
def extract_keypoints_from_images(image_paths, label, pose_model, max_images=2000):
    data = []
    paths = image_paths[:max_images]
    skipped = 0

    for i, p in enumerate(paths):
        if i % 100 == 0:
            print(f'  [{i}/{len(paths)}] {len(data)} keypoint sets so far...')

        img = cv2.imread(p)
        if img is None:
            skipped += 1
            continue

        results = pose_model(img, verbose=False)
        for r in results:
            if r.keypoints is None or r.boxes is None:
                continue
            kps_all = r.keypoints.data.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()

            for idx, kps in enumerate(kps_all):
                if kps.shape[0] != 17:
                    continue
                kps_xy = kps[:, :2]
                if np.sum(np.any(kps_xy > 0, axis=1)) < 10:
                    continue
                if idx < len(confs) and confs[idx] < 0.3:
                    continue
                data.append((kps_xy, label))

    print(f'  DONE: {len(data)} keypoint sets from {len(paths)} images (skipped {skipped})')
    return data


np.random.shuffle(ok_paths)
np.random.shuffle(injured_paths)

print('Extracting OK keypoints...')
ok_data = extract_keypoints_from_images(ok_paths, label=0, pose_model=pose_model)

print('\nExtracting INJURED keypoints...')
injured_data = extract_keypoints_from_images(injured_paths, label=1, pose_model=pose_model)

real_data = ok_data + injured_data
print(f'\n=== Real data: {len(real_data)} total (OK: {len(ok_data)}, INJURED: {len(injured_data)}) ===')

assert len(real_data) >= 50, \
    f'Only {len(real_data)} keypoint samples extracted. Check images contain visible people.'

## 4. Synthetic Augmentation

Supplement real data with synthetic keypoints to boost robustness and balance classes.

In [ ]:
def _make_kps(template, n, label, cx_range=(100,540), cy_range=(200,450), noise_std=5, lean_std=0.15):
    samples = []
    for _ in range(n):
        cx = np.random.uniform(*cx_range)
        cy = np.random.uniform(*cy_range)
        scale = np.random.uniform(0.7, 1.3)
        noise = np.random.normal(0, noise_std, (17, 2))
        a = np.random.normal(0, lean_std)
        cos_a, sin_a = np.cos(a), np.sin(a)
        rot = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
        kps = (np.array(template, dtype=float) @ rot.T) * scale + [cx, cy] + noise
        samples.append((kps, label))
    return samples

STANDING_T = [[0,-180],[-8,-190],[8,-190],[-15,-180],[15,-180],[-30,-140],[30,-140],
              [-40,-80],[40,-80],[-35,-20],[35,-20],[-20,0],[20,0],[-22,80],[22,80],[-22,160],[22,160]]

SITTING_T = [[0,-100],[-8,-108],[8,-108],[-15,-100],[15,-100],[-30,-65],[30,-65],
             [-45,-20],[45,-20],[-40,10],[40,10],[-20,0],[20,0],[-30,50],[30,50],[-25,10],[25,10]]

def _lying_t():
    d = np.random.choice([-1, 1])
    return [[d*-180,0],[d*-188,-8],[d*-188,8],[d*-178,-15],[d*-178,15],
            [d*-135,-25],[d*-135,25],[d*-75,-35],[d*-75,35],[d*-15,-30],
            [d*-15,30],[d*0,-18],[d*0,18],[d*80,-20],[d*80,20],[d*155,-20],[d*155,20]]

def gen_lying(n):
    return [s for _ in range(n) for s in _make_kps(_lying_t(), 1, 1, cy_range=(300,450), noise_std=6, lean_std=0.2)]

n_ok, n_inj = len(ok_data), len(injured_data)
synth_ok = max(100, int(0.15 * n_ok))
synth_inj = max(100, int(0.15 * n_inj))

if n_ok > 0 and n_inj > 0 and max(n_ok, n_inj) / max(min(n_ok, n_inj), 1) > 2:
    if n_ok > n_inj: synth_inj = max(synth_inj, int(0.3 * (n_ok - n_inj)))
    else: synth_ok = max(synth_ok, int(0.3 * (n_inj - n_ok)))

synth_data = _make_kps(STANDING_T, synth_ok, 0) + _make_kps(SITTING_T, synth_ok//2, 0) + gen_lying(synth_inj)

combined = real_data + synth_data
np.random.shuffle(combined)

pct = len(real_data) / len(combined) * 100
print(f'Combined: {len(combined)} samples ({pct:.0f}% real, {100-pct:.0f}% synthetic)')
print(f'  OK: {sum(1 for _,l in combined if l==0)}')
print(f'  INJURED: {sum(1 for _,l in combined if l==1)}')

## 5. Feature Engineering (11 features)

In [ ]:
def extract_features(keypoints):
    kps = np.array(keypoints)
    if kps.shape[-1] == 3:
        kps = kps[:, :2]

    nose = kps[0]
    l_shoulder, r_shoulder = kps[5], kps[6]
    l_hip, r_hip = kps[11], kps[12]
    l_knee, r_knee = kps[13], kps[14]
    l_ankle, r_ankle = kps[15], kps[16]

    mid_shoulder = (l_shoulder + r_shoulder) / 2
    mid_hip = (l_hip + r_hip) / 2
    mid_knee = (l_knee + r_knee) / 2
    mid_ankle = (l_ankle + r_ankle) / 2

    x_min, y_min = kps.min(axis=0)
    x_max, y_max = kps.max(axis=0)
    bbox_w = max(x_max - x_min, 1)
    bbox_h = max(y_max - y_min, 1)

    aspect_ratio = bbox_h / bbox_w
    torso_vec = mid_hip - mid_shoulder
    torso_angle = np.arctan2(torso_vec[0], torso_vec[1])
    body_vec = mid_ankle - nose
    body_angle = np.arctan2(body_vec[0], body_vec[1])
    span_ratio = bbox_h / bbox_w if bbox_w > 0 else 0
    nose_hip_vert = abs(nose[1] - mid_hip[1]) / bbox_h
    shoulder_ankle_vert = abs(mid_shoulder[1] - mid_ankle[1]) / bbox_h
    horiz_spread = bbox_w / bbox_h if bbox_h > 0 else 0
    y_variance = np.std(kps[:, 1]) / bbox_h if bbox_h > 0 else 0
    leg_vec = mid_ankle - mid_hip
    leg_angle = np.arctan2(leg_vec[0], leg_vec[1])
    body_center_y = (y_min + y_max) / 2
    head_relative = (body_center_y - nose[1]) / bbox_h
    knee_ankle_vert = abs(mid_knee[1] - mid_ankle[1]) / bbox_h

    return np.array([
        aspect_ratio, abs(torso_angle), abs(body_angle), span_ratio,
        nose_hip_vert, shoulder_ankle_vert, horiz_spread, y_variance,
        abs(leg_angle), head_relative, knee_ankle_vert,
    ])


FEATURE_NAMES = [
    'aspect_ratio', 'torso_angle', 'body_angle', 'span_ratio',
    'nose_hip_vert', 'shoulder_ankle_vert', 'horiz_spread', 'y_variance',
    'leg_angle', 'head_relative', 'knee_ankle_vert',
]

X_kps = np.array([kp for kp, _ in combined])
y = np.array([label for _, label in combined])
X_features = np.array([extract_features(kp) for kp in X_kps])

print(f'Feature matrix: {X_features.shape}')
print(f'Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')

In [ ]:
ncols = 4
nrows = (len(FEATURE_NAMES) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
for i, name in enumerate(FEATURE_NAMES):
    ax = axes.flatten()[i]
    ax.hist(X_features[y == 0, i], bins=40, alpha=0.6, label='OK', color='green')
    ax.hist(X_features[y == 1, i], bins=40, alpha=0.6, label='INJURED', color='red')
    ax.set_title(name)
    ax.legend()
for i in range(len(FEATURE_NAMES), len(axes.flatten())):
    axes.flatten()[i].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Train & Evaluate

In [ ]:
valid = np.isfinite(X_features).all(axis=1)
X_clean, y_clean = X_features[valid], y[valid]
print(f'Removed {(~valid).sum()} invalid rows. Using {len(X_clean)} samples.')

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print(f'Train balance — OK: {(y_train==0).sum()}, INJURED: {(y_train==1).sum()}')

In [ ]:
clf = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.9, random_state=42,
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['OK', 'INJURED']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
cv_acc = cross_val_score(clf, X_clean, y_clean, cv=5, scoring='accuracy')
cv_f1 = cross_val_score(clf, X_clean, y_clean, cv=5, scoring='f1')
print(f'5-Fold CV Accuracy: {cv_acc.mean():.3f} (+/- {cv_acc.std():.3f})')
print(f'5-Fold CV F1 (INJURED): {cv_f1.mean():.3f} (+/- {cv_f1.std():.3f})')

In [ ]:
importances = clf.feature_importances_
order = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 5))
plt.bar(range(len(FEATURE_NAMES)), importances[order], color='steelblue')
plt.xticks(range(len(FEATURE_NAMES)), [FEATURE_NAMES[i] for i in order], rotation=45, ha='right')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
os.makedirs('model', exist_ok=True)
joblib.dump(clf, 'model/injury_classifier.pkl')

config = {
    'feature_names': FEATURE_NAMES,
    'n_features': len(FEATURE_NAMES),
    'n_train_samples': int(len(X_train)),
    'n_real_samples': int(len(real_data)),
    'n_synthetic_samples': int(len(synth_data)),
    'cv_accuracy': float(cv_acc.mean()),
    'cv_f1': float(cv_f1.mean()),
}
with open('model/feature_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved model/injury_classifier.pkl')
print('Saved model/feature_config.json')
print(json.dumps(config, indent=2))

## 8. End-to-End Visual Test

In [ ]:
def classify_frame(frame, pose_model, classifier):
    results = pose_model(frame, verbose=False)
    detections = []
    for r in results:
        if r.keypoints is None or r.boxes is None:
            continue
        kps_all = r.keypoints.data.cpu().numpy()
        boxes = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        for idx, (kps, box) in enumerate(zip(kps_all, boxes)):
            if kps.shape[0] != 17: continue
            kps_xy = kps[:, :2]
            if np.sum(np.any(kps_xy > 0, axis=1)) < 8: continue
            if idx < len(confs) and confs[idx] < 0.3: continue
            features = extract_features(kps_xy).reshape(1, -1)
            if not np.isfinite(features).all(): continue
            pred = classifier.predict(features)[0]
            prob = classifier.predict_proba(features)[0]
            conf = prob[int(pred)]
            label = 'INJURED' if pred == 1 else 'OK'
            color = (0, 0, 255) if pred == 1 else (0, 200, 0)
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            text = f'{label} {conf:.0%}'
            (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
            cv2.rectangle(frame, (x1, y1-th-14), (x1+tw+8, y1), color, -1)
            cv2.putText(frame, text, (x1+4, y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
            for kp in kps_xy:
                cx, cy = int(kp[0]), int(kp[1])
                if cx > 0 and cy > 0:
                    cv2.circle(frame, (cx, cy), 4, color, -1)
            detections.append({'label': label, 'confidence': float(conf)})
    return frame, detections

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, path in enumerate(ok_paths[:3]):
    img = cv2.imread(path)
    if img is not None:
        ann, dets = classify_frame(img.copy(), pose_model, clf)
        axes[0, i].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        axes[0, i].set_title(f'Expected: OK | Got: {[d["label"] for d in dets]}')
    axes[0, i].axis('off')
for i, path in enumerate(injured_paths[:3]):
    img = cv2.imread(path)
    if img is not None:
        ann, dets = classify_frame(img.copy(), pose_model, clf)
        axes[1, i].imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
        axes[1, i].set_title(f'Expected: INJURED | Got: {[d["label"] for d in dets]}')
    axes[1, i].axis('off')
plt.suptitle('End-to-End Test', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: upload your own test image
from google.colab import files
print('Upload a test image (or skip):')
try:
    uploaded = files.upload()
    for name in uploaded:
        img = cv2.imread(name)
        if img is not None:
            ann, dets = classify_frame(img.copy(), pose_model, clf)
            print(f'{name}: {dets}')
            cv2_imshow(ann)
except:
    print('Skipped.')

## 9. Download Model

Download and place in `cv_model/model/` in your project, then run `python inference.py`

In [ ]:
from google.colab import files
files.download('model/injury_classifier.pkl')
files.download('model/feature_config.json')
print('Place both files in cv_model/model/ then run: python inference.py')